In [ ]:
from scipy.stats import gaussian_kde, pearsonr
import pandas as pd
import numpy as np
import os

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": True,
    "axes.linewidth": 2,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": "grey",
    "grid.alpha": 0.3,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "xtick.minor.size": 0,
    "ytick.minor.size": 0,
    "xtick.major.pad": 10,
    "ytick.major.pad": 6,
    "patch.edgecolor": "black",
    "patch.force_edgecolor": True,
})

In [ ]:
health_condition_palette = {
    "fit": "#009E73",
    "at-risk": "#E69F00",
    "unhealthy": "#E24B4A",
}

gender_palette = {
    "male": "#378ADD",
    "female": "#D4537E",
    "other": "#7F77DD",
}

stress_level_palette = {
    "low": "#639922",
    "medium": "#BA7517",
    "high": "#E24B4A",
}

sleep_quality_palette = {
    "poor": "#E24B4A",
    "average": "#BA7517",
    "good": "#639922",
}

physical_activity_level_palette = {
    "sedentary": "#B5D4F4",
    "moderate": "#378ADD",
    "active": "#185FA5",
}

diet_type_palette = {
    "non-veg": "#C0504D",
    "balanced": "#AC8A53",
    "veg": "#97C459",
}

smoking_alcohol_palette = {
    "no": "#639922",
    "occasional": "#BA7517",
    "yes": "#E24B4A",
}

In [ ]:
def annotate_counts(ax, total):
    for p in ax.patches:
        if (count := p.get_height()) == 0:
            continue
        ax.annotate(
            f"{int(count)}({count / total:.1%})",
            (p.get_x() + p.get_width() / 2, count),
            ha="center", va="bottom", xytext=(0, 3), textcoords="offset points",
        )

def style_axis(ax):
    ax.spines["left"].set_visible(False)
    return ax

def plot_count(
    data_frame, col, palette, order=None, ax=None,
    yticks=None, hue=None, hue_order=None, annotate=True, legend=False,
):
    order = order or list(palette.keys())
    sns.countplot(
        x=col, data=data_frame, order=order,
        hue=hue or col, hue_order=hue_order, palette=palette,
        legend=bool(legend), ax=ax,
    )
    if annotate:
        annotate_counts(ax, len(data_frame))
    if yticks is not None:
        ax.set_yticks(yticks)
    if legend:
        ax.legend(title=legend)
    return style_axis(ax)


def numeric_pair_grid(
    data, features, hue="health_condition", palette=health_condition_palette,
    hue_order=None, sample=False, figsize=None, seed=0,
):
    n = len(features)
    plot_df = data.sample(min(sample, len(data)), random_state=seed) if sample else data
    fig, axes = plt.subplots(n, n, figsize=figsize or (2.6 * n, 2.6 * n))

    for i, yf in enumerate(features):
        for j, xf in enumerate(features):
            ax = axes[i, j]
            if i == j:
                sns.kdeplot(x=xf, data=data, hue=hue, hue_order=hue_order,
                            palette=palette, fill=True, alpha=0.6, legend=False, ax=ax)
            else:
                sns.scatterplot(x=xf, y=yf, data=plot_df, hue=hue, hue_order=hue_order,
                                palette=palette, s=8, alpha=0.4, edgecolor="none",
                                legend=False, ax=ax)
                r, _ = pearsonr(*data[[xf, yf]].dropna().to_numpy().T)
                ax.legend([f"$r = {r:.2f}$"], handlelength=0, handletextpad=0,
                          loc="upper right", fontsize=8, framealpha=0.7)
            ax.set_xlabel(xf if i == n - 1 else "")
            ax.set_ylabel(yf if j == 0 else "")
            ax.tick_params(labelsize=8)

    order = hue_order or list(data[hue].dropna().unique())
    handles = [plt.Line2D([0], [0], marker="o", ls="", color=palette[c], label=c)
               for c in order]
    fig.legend(handles=handles, title=hue, loc="upper right", bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    return fig


def plot_numeric_dist(
    data_frame, col, ax, hue=None, hue_order=None,
    palette=None, show_mean_line=True, alpha=0.9,
):
    if hue is None:
        mu, sigma = data_frame[col].mean(), data_frame[col].std()
        labels = [f"$\\mu={mu:.2f}$, $\\sigma={sigma:.2f}$"]
    else:
        hue_order = hue_order or list(data_frame[hue].dropna().unique())
        stats = data_frame.groupby(hue)[col].agg(["mean", "std"])
        labels = [f"{c}  ($\\mu={stats.loc[c, 'mean']:.2f}$, $\\sigma={stats.loc[c, 'std']:.2f}$)"
                  for c in hue_order]

    sns.kdeplot(x=col, data=data_frame, hue=hue,
                hue_order=hue_order if hue else None,
                palette=palette if hue else None,
                fill=True, alpha=alpha, edgecolor="black", ax=ax)

    if hue is None:
        if show_mean_line:
            peak = gaussian_kde(data_frame[col].dropna())(mu)[0]
            ax.vlines(mu, 0, peak, color="black", ls="--", lw=1.5)
        ax.legend(labels, handlelength=0, handletextpad=0)
    else:
        ax.legend(ax.get_legend().legend_handles, labels, title=hue)

    return style_axis(ax)

## Table of contents

[§1. Overview](#§1.-Overview)

[§2. Data](#§2.-Data)
- [§2.1 Target `health_condition`](#§2.1-Target-health_condition)
- [§2.2 `gender`](#§2.2-gender)
- [§2.3 `sleep_duration`](#§2.3-sleep_duration)
- [§2.4 `heart_rate`](#§2.4-heart_rate)
- [§2.5 `bmi`](#§2.5-bmi)
- [§2.6 `calorie_expenditure`](#§2.6-calorie_expenditure)
- [§2.7 `step_count`](#§2.7-step_count)
- [§2.8 `exercise_duration`](#§2.8-exercise_duration)
- [§2.9 `water_intake`](#§2.9-water_intake)
- [§2.10 `stress_level`](#§2.10-stress_level)
- [§2.11 `sleep_quality`](#§2.11-sleep_quality)
- [§2.12 `physical_activity_level`](#§2.12-physical_activity_level)
- [§2.13 `diet_type`](#§2.13-diet_type)
- [§2.14 `smoking_alcohol`](#§2.14-smoking_alcohol)
- [§2.15 Correlation](#§2.15-Correlation)
- [§2.16 Missing values (per column and per sample)](#§2.16-Missing-values-(per-column-and-per-sample))
- [§2.17 Duplicates](#§2.17-Duplicates)

[§3. Modeling](#§3.-Modeling)
- [§3.1 Metric](#§3.1-Metric)
- [§3.2 Cross-validation](#§3.2-Cross-validation)
- [§3.3 Ensemble](#§3.3-Ensemble)
- [§3.4 Submission](#§3.4-Submission)

![Playground Series - Season 6 Episode 7 Predicting Student Health Risk image](https://www.kaggle.com/competitions/125223/images/header)

# §1. Overview

This notebook goes through exploratory data analysis (EDA) and modeling for the [Playground Series - Season 6 Episode 7 - Predicting Student Health Risk competition](https://www.kaggle.com/competitions/playground-series-s6e7/overview).

The goal of the competition is to predict each student's health risk (`health_condition`) — a multi-class classification task with three levels: `fit`, `at-risk`, and `unhealthy` — based on their lifestyle, physiological, and psychological attributes.

The dataset (both train and test) was inspired by the College Student Health Behavior Dataset and synthesised.

Note: Feature distributions are close to, but not exactly the same as, the original (source: [Dataset Description](https://www.kaggle.com/competitions/playground-series-s6e7/data))

# §2. Data

The competition dataset (train and test) was synthesized from a deep learning model inspired by and trained on the [College Student Health Behavior Dataset](https://www.kaggle.com/datasets/ziya07/college-student-health-behavior-dataset), so feature distributions are close to, but not identical to, the original. That original dataset is described as a comprehensive student health dataset of 50,000 records, built to capture the wide-ranging lifestyle, physiological, and psychological characteristics of college students. It is modeled on trends from large-scale student health studies conducted in China, giving realistic variation in daily habits and health-related attributes, and it integrates information from surveys, wearable devices, and institutional health records — spanning lifestyle behaviors, psychological indicators, and time-series data.

Column descriptions:

* `id` — identifier;
* `sleep_duration` — hours of sleep per observation;
* `heart_rate` — measured heart rate (bpm);
* `bmi` — body mass index;
* `calorie_expenditure` — calories burned;
* `step_count` — number of steps taken;
* `exercise_duration` — time spent exercising;
* `water_intake` — daily water consumption;
* `diet_type` — category of diet (`veg`, `non-veg`, `balanced`) followed;
* `stress_level` — reported level of stress (`low`, `medium`, `high`);
* `sleep_quality` — self-reported quality of sleep (`poor`, `average`, `good`);
* `physical_activity_level` — overall level of physical activity (`sedentary`, `moderate`, `active`);
* `smoking_alcohol` — smoking and alcohol habit category (`no`, `occasional`, `yes`);
* `gender` — gender (`male`, `female`, `other`);
* `health_condition` — target variable; student health class (`fit`, `at-risk`, `unhealthy`);

Source: [College Student Health Behavior Dataset](https://www.kaggle.com/datasets/ziya07/college-student-health-behavior-dataset)

In [ ]:
data_dir = "/kaggle/input/competitions/playground-series-s6e7"
train_path = os.path.join(data_dir, "train.csv")
train = pd.read_csv(train_path)

test_path = os.path.join(data_dir, "test.csv")
test = pd.read_csv(test_path)

sample_submission_path = os.path.join(data_dir, "sample_submission.csv")
sample_submission = pd.read_csv(sample_submission_path)

In [ ]:
train

In [ ]:
general_stats = train.describe(include="all").transpose()
general_stats

From the description and a first look at the data, we can group the columns by their nature:

* Identifier: `id`
* Numerical features: `sleep_duration`, `heart_rate`, `bmi`, `calorie_expenditure`, `step_count`, `exercise_duration`, `water_intake`
* Ordinal features: `stress_level`, `sleep_quality`, `physical_activity_level`
* Categorical features: `diet_type`, `smoking_alcohol`, `gender`
* Target: `health_condition`

Based on this classification we can set up appropriate preprocessing for each group (e.g. scaling for numerical features, order-aware encoding for ordinal features, and categorical encoding (One-Hot, Target encoding, etc.) for categorical features).

In [ ]:
id_column = "id"
numerical_features = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure", 
    "step_count", "exercise_duration", "water_intake",
]
ordinal_features = ["stress_level", "sleep_quality", "physical_activity_level"]
categorical_features = ["diet_type", "smoking_alcohol", "gender"]
target_column = "health_condition"

## §2.1 Target `health_condition`

The dataset includes a target variable, `health_condition`, which classifies students into three categories:

* `fit` – representing balanced and healthy conditions
* `at-risk` – indicating moderate health concerns
* `unhealthy` – representing higher levels of health risk

Source: [College Student Health Behavior Dataset](https://www.kaggle.com/datasets/ziya07/college-student-health-behavior-dataset)

In [ ]:
figure = plt.figure()
ax = figure.add_subplot()
plot_count(
    data_frame=train, 
    col="health_condition", 
    palette=health_condition_palette,
    ax=ax, 
    yticks=range(0, 600_000, 75_000),
)
figure.show()

**Insights**

* Class imbalance. The target is dominated by `at-risk` (85.9%), while `fit` (5.8%) and unhealthy `(8.4%)` are small minorities — a roughly 15:1 majority-to-minority ratio. Stratified splitting and cross-validation with probability calibration are essential so the minority classes aren't washed out. 

## §2.2 `gender`

In [ ]:
feature = "gender"
feature_palette = gender_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Roughly balanced gender split. `male` (34.5%), `female` (32.5%), and `other` (30.0%) are fairly even, with only a mild skew toward male — no gender dominates the data.
* Health-condition mix is consistent across genders. Each gender shows the same pattern — at-risk overwhelmingly dominant, with small, similar fit and unhealthy counts — so the target imbalance holds within every gender group rather than being driven by one.

## §2.3 `sleep_duration`

In [ ]:
feature = "sleep_duration"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Overall sleep is centered on a healthy ~7 hours. The pooled distribution is roughly bell-shaped with $\mu$ = 6.99, $\sigma$ = 1.22, spanning about 3–10 hours — a plausible, well-behaved feature with no extreme skew.
* `sleep_duration` separates the classes clearly — the standout among features so far. `unhealthy` students sleep least ($\mu$ = 5.37), `at-risk` sit in the middle ($\mu$ = 7.09), and `fit` sleep most ($\mu$ = 7.95), a clean monotonic ordering that matches the ordinal target (`unhealthy` → `at-risk` → `fit`).

## §2.4 `heart_rate`

In [ ]:
feature = "heart_rate"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Heart rate is a clean, symmetric bell centered on ~75 bpm ($\mu$ = 75.10, $\sigma$ = 8.18), spanning roughly 50–100 — a normal, physiologically plausible range with no skew or outlier tails, so it's a well-behaved feature needing no transformation.

* The three classes overlap almost entirely. All health conditions peak around the same ~75 bpm with near-identical spread, so unlike `sleep_duration`, `heart_rate` shows little standalone separating power — it likely contributes mainly through interactions with other features rather than on its own.

## §2.5 `bmi`

Body mass index (BMI) is a value derived from the mass (weight) and height of a person. BMI is calculated as the body mass, in kilograms ($\text{kg}$), divided by the square of the body height, in square metres ($\text{m}^2$); although the quotient has units of kilograms per square metre, BMI is most often reported normalized by 1 $\text{kg}/\text{m}^2$, thus as a pure number.

$$
\text{BMI} = \frac{\text{mass}_{\text{kg}}}{\text{height}_{\text{m}}^2}
$$

Source: [Body mass index](https://en.wikipedia.org/wiki/Body_mass_index)

In [ ]:
feature = "bmi"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* BMI is a tight, symmetric bell centered at $\mu = 22.98$ ($\sigma = 2.48$), sitting squarely in the healthy range (~18.5–25) and spanning roughly 15–32 with only a slight right tail — a clean feature with no problematic skew.

* The classes overlap heavily, but there's a subtle signal: `unhealthy` extends a bit further into the higher-BMI tail (past ~27) than `fit`, while `fit` leans slightly lower, so BMI carries mild separating power at the extremes even though the bulk of all three classes coincides around 23.

## §2.6 `calorie_expenditure`

In [ ]:
feature = "calorie_expenditure"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Calorie expenditure is bimodal — a small secondary peak near ~1800 and a dominant one around ~2200 — so the single summary ($\mu = 2226.08$, $\sigma = 347.53$) hides real structure; the two humps likely reflect distinct activity subpopulations and may warrant a derived binary/binned feature.

* The class split follows the overall shape closely, with `at-risk` tracing both humps, so calorie expenditure alone separates the classes only weakly — though the smaller ~1800 mode could carry some signal worth probing in interaction with activity-related features.

## §2.7 `step_count`

In [ ]:
feature = "step_count"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Step count is broad and roughly uniform (with a gentle rise toward higher counts) rather than bell-shaped, spanning ~500–15,500 with $\mu = 8615.95$ and a large $\sigma = 3929.40$ — the wide, flat spread means it splits students across a full activity range rather than clustering.

* This is one of the more separating features: `unhealthy` concentrates at the low end (few steps), `fit` shifts distinctly toward the high end (its density peaks past ~12,500), and `at-risk` fills the middle — a clear activity gradient that matches the ordinal target and complements the `sleep_duration` signal.

## §2.8 `exercise_duration`

In [ ]:
feature = "exercise_duration"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Exercise duration is a broad bell centered near ~40 min ($\mu = 38.75$, $\sigma = 14.74$) with a distinct spike at exactly 0 — a separate "does not exercise" subpopulation the smooth summary misses; a binary `exercises_at_all` flag would capture this cleanly.

* The classes separate along duration: `unhealthy` leans toward the low/zero end, `fit` shifts toward the higher durations (~40–70 min), and `at-risk` dominates the middle — a monotonic activity gradient consistent with `step_count` and the ordinal target, making this a useful feature.

## §2.9 `water_intake`

In [ ]:
feature = "water_intake"

figure = plt.figure(figsize=(13, 5))

ax = figure.add_subplot(1, 2, 1)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    ax=ax,
)

target_vs_feature_ax = figure.add_subplot(1, 2, 2)
plot_numeric_dist(
    data_frame=train,
    col=feature,
    hue="health_condition",
    hue_order=["fit", "unhealthy", "at-risk"],
    palette=health_condition_palette,
    ax=target_vs_feature_ax,
)

figure.tight_layout()
figure.show()

**Insights**

* Water intake is a fairly symmetric bell centered at ~2.2 L ($\mu = 2.19$, $\sigma = 0.52$), spanning roughly 0.5–4 L with a slight bump near ~3 L — a well-behaved feature with no problematic skew.

* The three classes overlap almost completely, all peaking together at ~2.2 L, so water intake carries little standalone separating power for `health_condition` and likely matters only in combination with other features.

## §2.10 `stress_level`

In [ ]:
feature = "stress_level"
feature_palette = stress_level_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Stress level is fairly balanced, skewed toward `medium` (37.9%), with `low` (24.3%) and `high` (25.8%) close behind — a well-populated ordinal feature with no rare category.

* Stress is strongly and cleanly separating — arguably the best categorical predictor so far. `fit` appears almost only at `low` stress, `unhealthy` almost only at `high`, and `medium` is essentially pure `at-risk`. Each class occupies a distinct stress band with little cross-over, mapping tightly onto the ordinal target (`low`→`fit`, `medium`→`at-risk`, `high`→`unhealthy`).

* This near-deterministic structure suggests stress level alone carries a large share of the signal — worth confirming it isn't a near-leaky proxy for the target, and it should interact well with the physiological features (sleep, activity).

## §2.11 `sleep_quality`

In [ ]:
feature = "sleep_quality"
feature_palette = sleep_quality_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Sleep quality is almost perfectly balanced across `poor` (30.7%), `average` (31.0%), and `good` (29.8%) — a uniform ordinal feature with no rare category.

* It separates the minority classes in the expected direction, but weakly. `at-risk` dominates every category (mirroring the overall imbalance), while within the minorities the gradient is clear: `unhealthy` is highest at `poor` and shrinks toward `good`, and `fit` does the opposite — rising from `poor` to `good`.

* The signal is directional but soft — every sleep-quality level is still overwhelmingly `at-risk`, so unlike `stress_level`, sleep quality nudges the minority probabilities rather than cleanly splitting the classes. Useful in combination, not decisive alone.

## §2.12 `physical_activity_level`

In [ ]:
feature = "physical_activity_level"
feature_palette = physical_activity_level_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Sleep quality is almost perfectly balanced across `poor` (30.7%), `average` (31.0%), and `good` (29.8%) — a uniform ordinal feature with no rare category.

* It separates the minority classes in the expected direction, but weakly. `at-risk` dominates every category (mirroring the overall imbalance), while within the minorities the gradient is clear: `unhealthy` is highest at `poor` and shrinks toward `good`, and `fit` does the opposite — rising from `poor` to `good`.

* The signal is directional but soft — every sleep-quality level is still overwhelmingly `at-risk`, so unlike `stress_level`, sleep quality nudges the minority probabilities rather than cleanly splitting the classes. Useful in combination, not decisive alone.

## §2.13 `diet_type`

In [ ]:
feature = "diet_type"
feature_palette = diet_type_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Diet type is evenly split across `non-veg` (32.6%), `balanced` (32.9%), and `veg` (33.5%) — a balanced categorical feature with no rare level.

* It carries essentially no separating power: the `fit`/`at-risk`/`unhealthy` proportions are nearly identical across all three diet types, each dominated by `at-risk` in the same ratio. Diet type looks non-predictive on its own and may only contribute marginally through interactions.

## §2.14 `smoking_alcohol`

In [ ]:
feature = "smoking_alcohol"
feature_palette = smoking_alcohol_palette

figure = plt.figure(figsize=(11, 5))

ax = figure.add_subplot(1, 2, 1)
plot_count(
    data_frame=train, 
    col=feature, 
    palette=feature_palette,
    ax=ax,
)

feature_vs_target_ax = figure.add_subplot(1, 2, 2)
plot_count(
    data_frame=train, 
    col=feature, 
    hue="health_condition",
    order=list(feature_palette.keys()),
    hue_order=list(health_condition_palette.keys()),
    palette=health_condition_palette,
    ax=feature_vs_target_ax,
    annotate=False,
    legend="health_condition",
)

figure.tight_layout()
figure.show()

**Insights**

* Smoking/alcohol is evenly distributed across `no` (31.8%), `occasional` (31.6%), and `yes` (32.4%) — a balanced ordinal feature with no rare level.

* It moves the minority classes in the expected direction, though modestly. Going from `no` → `occasional` → `yes`, `fit` steadily declines while `unhealthy` steadily rises; `at-risk` stays dominant and roughly flat throughout.

* The gradient is real but soft — every level is still overwhelmingly `at-risk`, so like `sleep_quality` this feature nudges the `fit`/`unhealthy` odds rather than cleanly separating classes. Useful in combination, not decisive alone.

## §2.15 Correlation

In [ ]:
figure = numeric_pair_grid(
    train, 
    numerical_features, 
    hue_order=list(health_condition_palette.keys()),
    sample=4000,
)
figure.show()

**Insights**

* Numerical features are almost entirely uncorrelated with each other — most pairwise Pearson coefficients sit at $r \approx 0.00$. The only exceptions form one small activity cluster: `calorie_expenditure`–`step_count` ($r = 0.40$), `calorie_expenditure`–`exercise_duration` ($r = 0.39$), and `step_count`–`exercise_duration` ($r = 0.44$) — all mild, all sensibly linking the movement-related measures. Everything else (sleep, heart rate, BMI, water intake) is effectively independent.

* Despite the near-zero linear correlations, the scatter panels show clear class structure that $r$ can't capture. Horizontal bands appear along `sleep_duration` — `unhealthy` points concentrate in a low-sleep stripe (~5–6 h) and `fit` in a high-sleep stripe — confirming sleep separates classes even though it doesn't correlate with other features. Vertical gaps/bands appear along `calorie_expenditure` (the bimodal ~1800 vs ~2200 split) and near `exercise_duration = 0` (the "no exercise" spike), reflecting the sub-populations seen in the 1-D plots.

## §2.16 Missing values (per column and per sample)

In [ ]:
missing_values_count = (train.isnull().sum(axis=0) / train.shape[0]) * 100
missing_values_count = missing_values_count.sort_values(ascending=False)

figure = plt.figure(figsize=(10, 5))
ax = figure.add_subplot()

sns.barplot(
    x=missing_values_count.index,
    y=missing_values_count.values,
    color="#E24B4A",
    legend=False,
    edgecolor="black",
    ax=ax,
    saturation=1,
)

ax.set_ylabel("Percentage")
ax.set_xlabel("")
ax.set_title("Missing values per column")
ax.tick_params(axis="x", rotation=90)

style_axis(ax)
figure.tight_layout()
figure.show()

**Insights**

* Missingness is present in nearly every feature but modest — the highest are `stress_level` (~12%) and `sleep_duration` (~11%), tapering down through the rest to ~1%, while `id` and the target `health_condition` have none (as they must).

* There's a striking pattern: the columns with the most missingness are exactly the strongest predictors. `stress_level`, `sleep_duration`, and `sleep_quality` — the features that separated the classes best — top the list, while weak/non-predictive features like `diet_type` and `exercise_duration` are the most complete. This suggests missingness may not be fully random, so how we handle it matters.

* No column is missing enough to drop. Imputation is straightforward (median for numeric, mode/"missing" category for categorical/ordinal), but because the informative columns are the ones with gaps, it's worth adding explicit "was-missing" indicator flags — the fact that a value is absent could itself carry signal. Tree models like LightGBM/XGBoost also handle NaNs natively, which is another reason they suit this dataset.

In [ ]:
na = train.isna()
na = na[na.mean().sort_values(ascending=False).index]

figure = plt.figure(figsize=(10, 6))
ax = figure.add_subplot()

sns.heatmap(
    na,
    cbar=False,
    cmap=["#EAEAEA", "#E24B4A"],
    yticklabels=False,
    ax=ax,
)
ax.set_title("Missing values")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

**Insights**

* Missingness looks scattered rather than block-structured — the red marks are spread fairly evenly down each column with no wide horizontal stripes cutting across many columns at once, so there's little sign that whole rows go missing together. Missing values appear largely independent across columns.

* The per-column density matches the bar chart: `stress_level` and `sleep_duration` are visibly the densest (occasional thick clusters of adjacent rows), the middle columns thin out, and `id`/`health_condition` are clean. The few short thick bands within a single column (e.g. runs in `stress_level`, `calorie_expenditure`) hint at small localized clusters, possibly tied to ordering, but they don't span columns.

## §2.17 Duplicates

In [ ]:
columns = set(train.columns) - {"id", "health_condition"}
duplicates = train[train.duplicated(subset=columns)]
duplicates

**Insights**
* There are no duplicates in training dataset.

# §3. Modeling

## §3.1 Metric

Submissions are scored using **balanced accuracy**, which compares the predicted class against the true target. Unlike plain accuracy, it weights every class equally regardless of size, making it well suited to this heavily imbalanced problem where `at-risk` dominates.

It is defined as the average of the per-class recall scores:

$$
\text{Balanced Accuracy} = \frac{1}{n} \sum_{i=1}^{n} \text{Recall}(C_i),
\qquad
\text{Recall}(C_i) = \frac{TP_i}{TP_i + FN_i}
$$

where $n$ is the number of classes and $\text{Recall}(C_i)$ is the recall for class $C_i$ — the fraction of that class's true members the model correctly identifies. Averaging recall across all classes means a model cannot score well simply by predicting the majority class; it must recognize the minority `fit` and `unhealthy` classes too.

This is available directly in scikit-learn as [`balanced_accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html).

In [ ]:
from sklearn.metrics import balanced_accuracy_score

## §3.2 Cross-validation


Because the `health_condition` target is imbalanced, a stratified splitting strategy is preferable. scikit-learn provides this through [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), which keeps each class's proportion consistent across every fold. The right validation strategy is crucial, especially in Kaggle competitions.

In [ ]:
from sklearn.model_selection import StratifiedKFold

num_folds = 5
shuffle = True
random_seed = 13

cv_strategy = StratifiedKFold(
    n_splits=num_folds, 
    shuffle=shuffle, 
    random_state=random_seed,
)
folds = list(cv_strategy.split(
    X=train, y=train[target_column].values, groups=None,
))

In [ ]:
compute_target_dist = lambda data, target_column: (data[target_column].value_counts() / data.shape[0]) * 100
general_target_dist = compute_target_dist(train, target_column)
dists = [general_target_dist]
for fold, (train_indices, val_indices) in enumerate(folds, 1):
    fold_train_target_dist = compute_target_dist(train.iloc[train_indices], target_column)
    fold_val_target_dist = compute_target_dist(train.iloc[val_indices], target_column)
    dists.extend([fold_train_target_dist, fold_val_target_dist])
dists = pd.DataFrame(
    dists,
    index=(
        ["general"] +
        [x
        for k in range(1, num_folds + 1)
        for x in (f"fold_{k}_train", f"fold_{k}_val")
        ]
    ),
)

class_order = ["fit", "at-risk", "unhealthy"]
train_max = 100 - 100 / num_folds
val_max = 100 / num_folds

fig = plt.figure(figsize=(14, 1.1 * (num_folds + 1) + 1))
gs = fig.add_gridspec(num_folds + 1, 2, width_ratios=[train_max, val_max])

ax_general = fig.add_subplot(gs[0, :])
axes = [[fig.add_subplot(gs[r, 0]), fig.add_subplot(gs[r, 1])] for r in range(1, num_folds + 1)]

def draw_bar(ax, row_name, total):
    left = 0.0
    for cls in class_order:
        pct = dists.loc[row_name, cls]
        w = pct / 100 * total
        ax.barh(0, w, left=left, height=0.6,
                color=health_condition_palette[cls], edgecolor="black")
        ax.text(left + w / 2, 0, f"{pct:.2f}%",
                    va="center", ha="center", fontsize=8)
        left += w
    ax.set_xlim(0, total)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.tick_params(length=0)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)

draw_bar(ax_general, "general", 100)
ax_general.set_ylabel("general", rotation=0, ha="right",
                      va="center", fontweight="bold", labelpad=25)
ax_general.set_title("full dataset", fontsize=13, fontweight="bold", loc="left")

for i in range(num_folds):
    draw_bar(axes[i][0], f"fold_{i + 1}_train", train_max)
    draw_bar(axes[i][1], f"fold_{i + 1}_val", val_max)
    axes[i][0].set_ylabel(f"fold #{i + 1}", rotation=0, ha="right",
                          va="center", fontweight="bold", labelpad=25)

axes[0][0].set_title("train", fontsize=13, fontweight="bold", loc="left")
axes[0][1].set_title("val", fontsize=13, fontweight="bold", loc="left")
axes[-1][0].set_xlabel("Percentage of dataset (%)")
axes[-1][1].set_xlabel("Percentage of dataset (%)")

handles = [Patch(facecolor=health_condition_palette[c], edgecolor="black", label=c)
           for c in class_order]
fig.legend(handles=handles, title="health_condition", ncol=3,
           loc="upper center", bbox_to_anchor=(0.5, 1.02), frameon=False)

fig.tight_layout()
fig.show()

## §3.3 Ensemble

For this first version, we'll use a simple blending ensemble: each base model predicts class probabilities, and the final prediction for every student is obtained by weighted averaging these probabilities and selecting the class with the highest combined probability.

In [ ]:
cols = ["fit", "at-risk", "unhealthy"]

test_ids = test["id"].values

preds_dir = "/kaggle/input/datasets/vad13irt/ps-s7e6-02072026"
kosprintr = pd.read_csv(os.path.join(preds_dir, "kosprintr_preds_0.95052.csv"))[cols]
yekenot_pl = pd.read_csv(os.path.join(preds_dir, "yekenot_preds_pl_0.95040.csv"))[cols]
yekenot = pd.read_csv(os.path.join(preds_dir, "yekenot_preds_ 0.95029.csv"))[cols]
kirill0212 = pd.read_csv(os.path.join(preds_dir, "kirill0212_preds_0.95024.csv"))[cols]
hmnshudhmn24 = pd.read_csv(os.path.join(preds_dir, "hmnshudhmn24_preds_0.95021.csv"))[cols]

blend = 0.6*kosprintr + 0.4*yekenot # *yekenot #+ 0*kirill0212 + 0*hmnshudhmn24
preds = [cols[p] for p in np.argmax(blend.values, axis=1)]

## §3.4 Submission


Once the model (or models) is trained, we can generate predictions on the test set and write them to a submission file in the required format, as laid out in the [Evaluation section](https://www.kaggle.com/competitions/playground-series-s6e7/overview/evaluation). 

For every `id` in the test set, predict one label — `at-risk`, `unhealthy`, or `fit` — for the `health_condition` target. The file needs a header row and should look like this:

In [ ]:
sample_submission

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "health_condition": preds,
})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

submission

In [ ]:
figure = plt.figure()
ax = figure.add_subplot()
plot_count(
    data_frame=submission, 
    col="health_condition", 
    palette=health_condition_palette,
    ax=ax, 
)
figure.show()

Then submitting on [Kaggle Leaderboard](https://www.kaggle.com/competitions/playground-series-s6e7/leaderboard)!

<!-- ![Submission Leaderboard results 01.07.2026](https://i.ibb.co/nXNWSqW/2026-07-01-15-56-22.png) -->
![Submission Leaderboard results 02.07.2026](https://i.ibb.co/6RbdNGpw/2026-07-02-14-12-33.png)

### TODO

* Optimize memory for data frames.1. 
* Add citations, references, and acknowledgments.
* Add more variety to the graphs (pie charts, box plots, etc.).